### Подготовка и проверка железа, установка ID для моторчиков

Проверка, на какой порт подключён манипулятор

In [ ]:
!lerobot-find-port

# Вывод должен быть следующий:
# Finding all available ports for the MotorBus.
# ['/dev/ttyACM3', '/dev/ttyACM2']
# Remove the USB cable from your MotorsBus and press Enter when done.

# [...Disconnect corresponding leader or follower arm and press Enter...]

# The port of this MotorsBus is /dev/ttyACM2
# Reconnect the USB cable.

Установка ID моторчиков

In [ ]:
from lerobot.robots.so_follower import SO101Follower, SO101FollowerConfig

config = SO101FollowerConfig(
    port="/dev/ACM2",
    id="my_awesome_follower_arm",
)
follower = SO101Follower(config)
follower.setup_motors()

### Калибровка робота

In [ ]:
from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower

config = SO101FollowerConfig(
    port="/dev/ttyACM2",
    id="my_awesome_follower_arm1",
)

follower = SO101Follower(config)
follower.connect(calibrate=False)
follower.calibrate()
follower.disconnect()

### Сбор датасета

#### Простой Teleop

In [ ]:
%%bash
lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM2 \
    --robot.id=my_awesome_follower_arm \
    --teleop.type=so101_leader \
    --teleop.port=/dev/ttyACM3 \
    --teleop.id=my_awesome_leader_arm \
    --display_data=true

In [ ]:
from lerobot.teleoperators.so_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower

robot_config = SO101FollowerConfig(
    port="/dev/ttyACM2",
    id="my_awesome_follower_arm",
)

teleop_config = SO101LeaderConfig(
    port="/dev/ttyACM3",
    id="my_awesome_leader_arm",
)

robot = SO101Follower(robot_config)
teleop_device = SO101Leader(teleop_config)
robot.connect()
teleop_device.connect()

while True:
    action = teleop_device.get_action()
    robot.send_action(action)

Скрипт чтобы узнать ID для камеры realsense

In [ ]:
import pyrealsense2 as rs

ctx = rs.context()
for dev in ctx.query_devices():
    print(
        dev.get_info(rs.camera_info.name),
        dev.get_info(rs.camera_info.serial_number)
    )

#### Запись Эпизодов

In [ ]:
!hf auth login --token hf_ZZZZZZZZZZZZZZZZZZZZZ --add-to-git-credential # Вместо hf_ZZZZZZZZZZZZZZZZZZZZZ вы должны вставить свой токен с HF.

In [ ]:
from lerobot.cameras.realsense.configuration_realsense import RealSenseCameraConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.teleoperators.so_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import init_rerun
from lerobot.scripts.lerobot_record import record_loop
from lerobot.processor import make_default_processors
from lerobot.cameras.configs import ColorMode, Cv2Rotation

NUM_EPISODES = 50
FPS = 30
EPISODE_TIME_SEC = 40
RESET_TIME_SEC = 10
TASK_DESCRIPTION = "Put the red cube into the white bucket "

# Create robot configuration
robot_config = SO101FollowerConfig(
    id="my_awesome_follower_arm",
    cameras={
        "front": RealSenseCameraConfig(
            serial_number_or_name = "231122072775", 
            fps = FPS, 
            width = 640, 
            height = 480, 
            rotation=Cv2Rotation.ROTATE_180) # Optional: fourcc="MJPG" for troubleshooting OpenCV async error.
    },
    port="/dev/ttyACM2",
)

teleop_config = SO101LeaderConfig(
    id="my_awesome_leader_arm",
    port="/dev/ttyACM3",
)

# Initialize the robot and teleoperator
robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

# Configure the dataset features
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}

# Create the dataset
dataset = LeRobotDataset.create(
    repo_id="MrAnton/so101_workshop",
    fps=FPS,
    features=dataset_features,
    robot_type=robot.name,
    use_videos=True,
    image_writer_threads=4,
)

# Initialize the keyboard listener and rerun visualization
_, events = init_keyboard_listener()
init_rerun(session_name="recording")

# Connect the robot and teleoperator
robot.connect()
teleop.connect()

# Create the required processors
teleop_action_processor, robot_action_processor, robot_observation_processor = make_default_processors()

episode_idx = 0
while episode_idx < NUM_EPISODES and not events["stop_recording"]:
    log_say(f"Recording episode {episode_idx + 1} of {NUM_EPISODES}")

    record_loop(
        robot=robot,
        events=events,
        fps=FPS,
        teleop_action_processor=teleop_action_processor,
        robot_action_processor=robot_action_processor,
        robot_observation_processor=robot_observation_processor,
        teleop=teleop,
        dataset=dataset,
        control_time_s=EPISODE_TIME_SEC,
        single_task=TASK_DESCRIPTION,
        display_data=True,
    )

    # Reset the environment if not stopping or re-recording
    if not events["stop_recording"] and (episode_idx < NUM_EPISODES - 1 or events["rerecord_episode"]):
        log_say("Reset the environment")
        record_loop(
            robot=robot,
            events=events,
            fps=FPS,
            teleop_action_processor=teleop_action_processor,
            robot_action_processor=robot_action_processor,
            robot_observation_processor=robot_observation_processor,
            teleop=teleop,
            control_time_s=RESET_TIME_SEC,
            single_task=TASK_DESCRIPTION,
            display_data=True,
        )

    if events["rerecord_episode"]:
        log_say("Re-recording episode")
        events["rerecord_episode"] = False
        events["exit_early"] = False
        dataset.clear_episode_buffer()
        continue

    dataset.save_episode()
    episode_idx += 1

# Clean up
log_say("Stop recording")
robot.disconnect()
teleop.disconnect()
dataset.finalize()
dataset.push_to_hub()

Отправка в HF

In [ ]:
!hf upload ${HF_USER}/record-test ~/.cache/huggingface/lerobot/MrAnton/so101_workshop/ --repo-type dataset 

#### Прямое воспроизведение датасета

In [3]:
import time

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.teleoperators.so_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower
from lerobot.utils.robot_utils import precise_sleep
from lerobot.utils.utils import log_say

episode_idx = 0

robot_config = SO101FollowerConfig(port="/dev/ttyACM2", id="my_awesome_follower_arm")

robot = SO101Follower(robot_config)
robot.connect()

dataset = LeRobotDataset("MrAnton/so101_workshop", episodes=[episode_idx])
actions = dataset.hf_dataset.select_columns("action")

log_say(f"Replaying episode {episode_idx}")
for idx in range(dataset.num_frames):
    t0 = time.perf_counter()

    action = {
        name: float(actions[idx]["action"][i]) for i, name in enumerate(dataset.features["action"]["names"])
    }
    robot.send_action(action)

    precise_sleep(max(1.0 / dataset.fps - (time.perf_counter() - t0), 0.0))

robot.disconnect()

/home/anton/miniconda3/envs/workshop_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:21<00:00, 10.99s/it]


In [7]:
dataset = LeRobotDataset("MrAnton/so101_workshop", episodes=[0])
row = dataset.hf_dataset[0]
print(row.keys())

dict_keys(['action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'])


In [ ]:
print(row["action"]) 

tensor([ -10.5055, -101.0549,   97.0549,   77.4066,   73.4505,    1.7500])


In [10]:
print(row["observation.state"])

tensor([ -7.1209, -97.8022,  96.1319,  69.2747,  60.7033,   2.1088])


In [12]:
row = dataset.hf_dataset[0]
print(row["timestamp"])
row = dataset.hf_dataset[1]
print(row["timestamp"])

tensor(0.)
tensor(0.0333)


### Обучение act policy на полученных данных

In [ ]:
%%bash
lerobot-train \
  --dataset.repo_id=MrAnton/so101_workshop \
  --policy.type=act \
  --output_dir=outputs/train/act_so101_cu_i_bu \
  --job_name=act_so101_cu_i_bu \
  --policy.device=cuda \
  --wandb.enable=false \
  --policy.repo_id=MrAnton/cube_into_buck_policy

*Огромный исходник train скрипта можно изучить в ~/lerobot/scripts/lerobot_train.py